# Example 21 — 2-D steady conduction: the heated-edge plate

The first PDE of every heat-transfer course:
$$\nabla^2 T = 0\ \text{on}\ [0,1]^2,\qquad T = \sin(\pi x)\ \text{on the top edge},\quad T=0\ \text{elsewhere},$$
with the separation-of-variables exact solution
$$T = \sin(\pi x)\,\frac{\sinh(\pi y)}{\sinh(\pi)}.$$

The PINN needs nothing new — Laplace residual + four soft Dirichlet edges — which is the
point: after the machinery of Examples 1–20, a textbook 2-D problem is a five-minute
exercise. Verified: L2 ≈ 1.6e-03 in ~50 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

def T_exact(x, y):
    return torch.sin(np.pi*x)*torch.sinh(np.pi*y)/np.sinh(torch.tensor(np.pi))

net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
opt = torch.optim.Adam(net.parameters(), 2e-3)
t0 = time.perf_counter()
for e in range(4000):
    opt.zero_grad()
    x = torch.rand(2000,1,device=device).requires_grad_(True)
    y = torch.rand(2000,1,device=device).requires_grad_(True)
    T = net(torch.cat([x,y],1))
    res = g1(g1(T,x),x) + g1(g1(T,y),y)
    s = torch.rand(300,1,device=device)
    loss = (res**2).mean() + 10*(
        ((net(torch.cat([s,torch.ones_like(s)],1)) - torch.sin(np.pi*s))**2).mean()
        + (net(torch.cat([s,torch.zeros_like(s)],1))**2).mean()
        + (net(torch.cat([torch.zeros_like(s),s],1))**2).mean()
        + (net(torch.cat([torch.ones_like(s),s],1))**2).mean())
    loss.backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s')

n = 101
xs = torch.linspace(0,1,n,device=device); X, Y = torch.meshgrid(xs, xs, indexing='ij')
with torch.no_grad():
    Tp = net(torch.cat([X.reshape(-1,1), Y.reshape(-1,1)],1)).reshape(n,n).cpu().numpy()
Te = T_exact(X.reshape(-1,1), Y.reshape(-1,1)).reshape(n,n).cpu().numpy()
print(f'L2 = {np.sqrt(np.mean((Tp-Te)**2)):.2e}')

fig, ax = plt.subplots(1, 2, figsize=(11.5,4.3))
c0 = ax[0].contourf(X.cpu(), Y.cpu(), Tp, 21, cmap='inferno'); plt.colorbar(c0, ax=ax[0])
ax[0].set_title('PINN temperature field'); ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[1].plot(Te[:, n//2], 'g', lw=2.2, label='exact (x=0.5 line)')
ax[1].plot(Tp[:, n//2], 'r--', lw=1.5, label='PINN')
ax[1].plot(Te[n//2, :], 'g', lw=2.2, alpha=.4, label='exact (y=0.5 line)')
ax[1].plot(Tp[n//2, :], 'r--', lw=1.5, alpha=.6)
ax[1].set_xlabel('grid index'); ax[1].set_ylabel('T'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
ax[1].set_title('Centre-line cuts')
plt.tight_layout(); plt.show()

## Observations
- **Exponential wall-off:** $\sinh(\pi y)/\sinh(\pi)$ — the heated edge's influence dies
  fast; most of the plate barely notices it (Saint-Venant in thermal form).
- **Same solver as Example 4** — this Laplacian is the d=2 case of the high-dimensional
  Poisson notebook; nothing in the code knows the dimension.
- **A good exam problem:** ask students to swap the heated edge or superpose two edges
  (linearity!) — one-line changes.

**Try:** Robin (convective) edges $-k\,\partial T/\partial n = h(T - T_\infty)$; an
interior heat source (Poisson); or a hole in the plate — mesh-free sampling makes the
last one trivial (Example 18's annulus trick).